In [0]:
import os
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ObjectType, ExportFormat, ImportFormat
w = WorkspaceClient()
catalog = dbutils.widgets.get("catalog")
schemas = dbutils.widgets.get("schemas")

# Replace placeholders in generated notebooks
for schema in schemas:
    output_folder = f"/Workspace/Users/depoplimontj@delawareconsulting.com/delaware_lakebridge/switch_runs/{schema}" 
    for obj in sorted(w.workspace.list(output_folder), key=lambda o: o.path):
        if obj.object_type == ObjectType.NOTEBOOK:
            export_resp = w.workspace.export(obj.path, format=ExportFormat.SOURCE)
            content = base64.b64decode(export_resp.content).decode("utf-8")
            content = content.replace("{catalog}", catalog).replace("{schema}", schema)
            w.workspace.import_(
                obj.path,
                content=base64.b64encode(content.encode("utf-8")).decode("utf-8"),
                format=ImportFormat.SOURCE,
                language=obj.language,
                overwrite=True,
            )

    # Run generated notebooks
    for obj in sorted(w.workspace.list(output_folder), key=lambda o: o.path):
        if obj.object_type == ObjectType.NOTEBOOK:
            dbutils.notebook.run(
                obj.path,
                timeout_seconds=100000,
                arguments={"catalog": catalog, "schema": schema},
            )


# ── 1. Read the log table ────────────────────────────────────────────────────
df = spark.table(f"dbe_dbx_internships.log.run_details")

failed_df = df.filter(df.run_status == "FAILED")
failed_rows = failed_df.select("input_file_path", "input_file_relative_path").collect()

print(f"Found {len(failed_rows)} FAILED file(s) to re-upload.")'''

# ── 2. Re-upload each failed file to a v2 folder ────────────────────────────
import os

results = []

for row in failed_rows:
    src_path = row["input_file_path"]          # e.g. /Volumes/cat/schema/vol/dbo/file.sql
    rel_path  = row["input_file_relative_path"] # e.g. dbo.SomeTable.Table.sql

    # Derive the destination path:
    # Replace the last path segment's parent folder name with <folder>_v2
    # src_path structure: /Volumes/<catalog>/<schema>/<volume>/<subfolder>/file.sql
    src_dir  = os.path.dirname(src_path)        # /Volumes/.../dbo
    vol_root = os.path.dirname(src_dir)         # /Volumes/.../<volume>
    subfolder = os.path.basename(src_dir)       # dbo
    dst_dir  = os.path.join(vol_root, f"{subfolder}_v2")  # /Volumes/.../<volume>/dbo_v2
    dst_path = os.path.join(dst_dir, os.path.basename(src_path))

    try:
        # Read source file from UC Volume
        with open(src_path, "rb") as f:
            content = f.read()

        # Ensure destination directory exists
        os.makedirs(dst_dir, exist_ok=True)

        # Write to destination
        with open(dst_path, "wb") as f:
            f.write(content)

        results.append({"file": rel_path, "dst": dst_path, "status": "OK"})
        print(f"  ✓  {rel_path}  →  {dst_path}")

    except Exception as e:
        results.append({"file": rel_path, "dst": dst_path, "status": f"ERROR: {e}"})
        print(f"  ✗  {rel_path}  →  ERROR: {e}")

# ── 3. Summary ───────────────────────────────────────────────────────────────
ok    = sum(1 for r in results if r["status"] == "OK")
error = sum(1 for r in results if r["status"] != "OK")
print(f"\nDone — {ok} re-uploaded, {error} error(s).")

if error:
    for r in results:
        if r["status"] != "OK":
            print(f"  FAILED: {r['file']} — {r['status']}")